In [0]:
# Imports

from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("RegisterDeltaTables").getOrCreate()

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+---------------------+
|catalog              |
+---------------------+
|dbw_aqi_lakehouse_dev|
|samples              |
|system               |
+---------------------+



In [0]:
spark.sql("USE CATALOG dbw_aqi_lakehouse_dev")

DataFrame[]

In [0]:
# Create Schema/Database

spark.sql("CREATE SCHEMA IF NOT EXISTS aqi_lakehouse")
spark.sql("USE SCHEMA aqi_lakehouse")

spark.sql("SHOW SCHEMAS").show()

+------------------+
|      databaseName|
+------------------+
|     aqi_lakehouse|
|           default|
|information_schema|
+------------------+



In [0]:
# Gold data paths

BASE_PATH = "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net"

GOLD_DELTA_BASE_PATH = f"{BASE_PATH}/gold_delta"

DAILY_CITY_AQI_PATH = f"{GOLD_DELTA_BASE_PATH}/daily_city_aqi/"
DAILY_CITY_RANKING_PATH = f"{GOLD_DELTA_BASE_PATH}/daily_city_ranking/"
DATA_FRESHNESS_PATH = f"{GOLD_DELTA_BASE_PATH}/data_freshness/"
DATA_COMPLETENESS_PATH = f"{GOLD_DELTA_BASE_PATH}/data_completeness/"

In [0]:
# Register Tables 

# Daily city aqi
spark.sql(f"""
CREATE TABLE IF NOT EXISTS dbw_aqi_lakehouse_dev.aqi_lakehouse.daily_city_aqi
USING DELTA
LOCATION '{DAILY_CITY_AQI_PATH}'
""")

# Daily city ranking
spark.sql(f"""
CREATE TABLE IF NOT EXISTS dbw_aqi_lakehouse_dev.aqi_lakehouse.daily_city_ranking
USING DELTA
LOCATION '{DAILY_CITY_RANKING_PATH}'
""")

# Data freshness
spark.sql(f"""
CREATE TABLE IF NOT EXISTS dbw_aqi_lakehouse_dev.aqi_lakehouse.data_freshness
USING DELTA
LOCATION '{DATA_FRESHNESS_PATH}'
""")

# Data completeness
spark.sql(f"""
CREATE TABLE IF NOT EXISTS dbw_aqi_lakehouse_dev.aqi_lakehouse.data_completeness
USING DELTA
LOCATION '{DATA_COMPLETENESS_PATH}'
""")

DataFrame[]

In [0]:
spark.sql("""
SELECT *
FROM dbw_aqi_lakehouse_dev.aqi_lakehouse.daily_city_aqi
ORDER BY measurement_date, city
""").show(truncate=False)

+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+
|city        |measurement_date|avg_european_aqi|max_european_aqi|avg_pm10|max_pm10|avg_pm2_5|max_pm2_5|avg_nitrogen_dioxide|max_nitrogen_dioxide|record_count|
+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+
|Athens      |2026-07-09      |39.36           |60              |17.19   |25.2    |12.0     |17.5     |36.0                |71.2                |22          |
|Berlin      |2026-07-09      |23.23           |30              |8.05    |11.0    |4.0      |4.1      |4.0                 |6.6                 |22          |
|London      |2026-07-09      |35.95           |63              |14.68   |16.9    |10.0     |10.6     |21.0                |39.9                |22          |
|Paris       |2026-07-09      |29.0           

In [0]:
# Show tables

spark.sql("SHOW TABLES IN aqi_lakehouse").show(truncate=False)

+-------------+------------------+-----------+
|database     |tableName         |isTemporary|
+-------------+------------------+-----------+
|aqi_lakehouse|daily_city_aqi    |false      |
|aqi_lakehouse|daily_city_ranking|false      |
|aqi_lakehouse|data_completeness |false      |
|aqi_lakehouse|data_freshness    |false      |
+-------------+------------------+-----------+



In [0]:
# SQL Checks

# Worst aqi per city

print("Worst aqi per city")
spark.sql("""
SELECT
    measurement_date,
    city,
    avg_european_aqi,
    rank
FROM aqi_lakehouse.daily_city_ranking
WHERE rank = 1
ORDER BY measurement_date
""").show(truncate=False)

# Freshness check

print("Data Freshness check")
spark.sql("""
SELECT *
FROM aqi_lakehouse.data_freshness
ORDER BY city
""").show(truncate=False)

# Completeness check

print("Completeness Check")
spark.sql("""
SELECT *
FROM aqi_lakehouse.data_completeness
ORDER BY measurement_date, city
""").show(truncate=False)

Worst aqi per city
+----------------+------------+----------------+----+
|measurement_date|city        |avg_european_aqi|rank|
+----------------+------------+----------------+----+
|2026-07-09      |Paris       |29.0            |1   |
|2026-07-09      |Berlin      |23.23           |1   |
|2026-07-09      |Athens      |39.36           |1   |
|2026-07-09      |London      |35.95           |1   |
|2026-07-09      |Thessaloniki|37.82           |1   |
+----------------+------------+----------------+----+

Data Freshness check
+------------+------------------------------+----------------------------+
|city        |latest_ingestion_timestamp_utc|latest_measurement_timestamp|
+------------+------------------------------+----------------------------+
|Athens      |2026-07-09 20:58:21.635676    |2026-07-09 21:00:00         |
|Berlin      |2026-07-09 20:58:23.578567    |2026-07-09 21:00:00         |
|London      |2026-07-09 20:58:22.555047    |2026-07-09 21:00:00         |
|Paris       |2026-07-0